# 6. Validation: In-Plane Shear and Inertia

The previous notebook validated the solver on typical *lubrication* problems, where the flow is governed by the gap shear (wall stresses $\tau_{xz}, \tau_{yz}$) and the pressure gradient. Here we do not check such problems. Instead, we test the following two contributions:

- **In-plane shear:** viscous momentum diffusion within the film plane.
- **Inertia:** the convective momentum flux.

To isolate them, the gap shear is switched off (`gap_shear: False`) and the cases are chosen such that an exact solution exists.

| Case | Term tested | Driven by | Exact solution |
|------|-------------|-----------|----------------|
| Poiseuille | in-plane shear | body force $f_x$ | in-plane parabolic profile |
| Couette | in-plane shear | moving wall | in-plane Couette profile |
| Venturi | inertia | fixed inlet flux | mass conservation + Bernoulli |

In the two shear cases, the no-slip walls sit at the cell faces $y=0$ and $y=L_y$, so the exact solution is evaluated at the cell centers. The setup is shown for each case: load the YAML, set a uniform density as initial state, run.

In [ ]:
from GaPFlow.problem import Problem
from utils.topography import regen_bernoulli_venturi
from utils.plotting import plot_poiseuille_validation, plot_couette_validation, plot_bernoulli_validation

## Poiseuille Flow

A body force $f_x$ drives flow between two stationary walls. The flow is periodic in $x$ and the momentum balance reduces to a balance between $f_x$ and in-plane viscous diffusion in $y$:

$$u(y) = \frac{\rho f_x}{2\mu}\, y\,(L_y - y)$$

The stationary walls are realized by Dirichlet boundary conditions on the mass flux on the northern/souther boundaries.

$\rightarrow$ **[YAML File](../../tests/configs/poiseuille_2d_body_force.yaml)**

In [ ]:
problem = Problem.from_yaml("../../tests/configs/poiseuille_2d_body_force.yaml")
problem.q[0][:] = 1.0
problem.run()
plot_poiseuille_validation(problem)

## Couette Flow

The bottom wall is at rest, the top wall moves with $U$ (Dirichlet BC $j_x = \rho U$). The flow is periodic in $x$ and the solution is a linear profile:

$$u(y) = U\,\frac{y}{L_y}$$

The stationary and moving walls are again realized by Diriclet boundary conditions on the mass flux.

$\rightarrow$ **[YAML File](../../tests/configs/couette_2d_wall_velocity.yaml)**

In [ ]:
problem = Problem.from_yaml("../../tests/configs/couette_2d_wall_velocity.yaml")
problem.q[0][:] = 1.0
problem.run()
plot_couette_validation(problem)

## Bernoulli Venturi Nozzle

Nearly incompressible flow through a symmetric contraction from $h_\mathrm{inlet} = 1\,\mathrm{mm}$ to $h_\mathrm{throat} = 0.5\,\mathrm{mm}$ tests the convective terms (`inertia: True`). The inlet flux is prescribed ($j_x = \rho v_\mathrm{inlet}$ with $v_\mathrm{inlet} = 5\,\mathrm{m/s}$), the outlet density is fixed. For constant density, inviscid flow and negligible body forces, the Bernoulli equation predicts:

$$v_\mathrm{throat} = v_\mathrm{inlet}\,\frac{h_\mathrm{inlet}}{h_\mathrm{throat}}, \qquad \Delta p = \tfrac{1}{2}\rho\,\left(v_\mathrm{throat}^2 - v_\mathrm{inlet}^2\right)$$

<img src="./figures/Geometry_BernoulliVenturi.png"
     alt="Bernoulli venturi geometry"
     width="250">

Note that under the assumption of inviscid flow, meaning plug flow, inertia can be exactly evaluated given height averaged density and mass fluxes.

$\rightarrow$ **[YAML File](../../tests/configs/bernoulli_venturi.yaml)**

In [ ]:
problem = Problem.from_yaml("../../tests/configs/bernoulli_venturi.yaml")
regen_bernoulli_venturi(problem)
problem.q[1][:] = 1000.0 * 5.0
problem.run()
plot_bernoulli_validation(problem)